Node executes --> returns
``` python
{
    "answer" : "Langgraph is ..."
}
```
Then Langgraph Merges --> New state. This normal flow easy.

Now imaginge in parllel execution.
``` markdown
          Node A
         /
Start ──┤
         \
          Node B
```

Here Node A and Node B run at the same time in parallel , this is called parllel excecution.

In [1]:
## For Example 
sate = {
    "documents" : [],
    "summary" : ""
}

# Node A returns

{
    "documents" : [
        "Doc 1",
        "Doc 2"
    ]
}

# Node B returns
{
    "summary":
    "artificial intelligence..."
}

# can langgraph merge these? yes result will become

updated_state = {
    "document" : [
        "Doc 1",
        "Doc 2"
    ],
    "summary" : "artificial intelligence"
}
# different fields no confilct , everything works 

In [2]:
# Now lets create a problem to understnad the reducers.
# current state
{
    "messages" : []
}

# node A returns 
{
    "messages" : [
        "hello"
    ]
}

# Node B returns
{
    "messages" :[
        "world"
    ]
}

# now if we observe here Both nodes are updated the "messages" which one langgrah should keep.

# option 1
{
    "messages": [
        "Hello"
    ]
}

# option 2
{
    "messages": [
        "World"
    ]
}

# option 3 : 
{
    "messages": [
        "Hello",
        "World"
    ]
}

# out of all the above states , which one is correct? Langgraph doesnot know.
# this is the reason reducers Exist : Reducers tells langgraph how to combine multiple updates to the same
# State field. 
# reducer are not about State, nodes, its about Conflicts.


{'messages': ['Hello', 'World']}

#### Reducers in LangGraph
A Reducer tells LangGraph how to combine multiple updates to the same State field.
Reducer are not about State, nodes, its about Conflicts.
it resolves the Conflicts and tells the langgraph to updated the multiple updated states to the fields.



##### Real life analogy
* Two teachers updating the marks , math teacher  updates m = 95 and science teacher update m = 65. it is easy different subjects. now imagine , there are two teachers for mathematics teacher a updates m = 95 and teacher b updates m = 98 , school asks which one should we keep? that decision maker is like a Reducer. 

* State Merging only helpful only one updates to next node , so if nodes are running in parallel and langgraph should update the state , for that reducer will combine the results and updates to the fields 

In [3]:
## Example : current satte
state = {
    "messages": [
        "Hi"
    ]
}

# update
update = {

    "messages": [

        "Hello"

    ]

}

# here without reducer 
{
"messages":[
"Hello"
]
} # here only updated 

# with append reducer 
{
"messages":[
"Hi",
"Hello"
]
}

{'messages': ['Hi', 'Hello']}

In [4]:
### Reducers in Langgraph
from typing import TypedDict
from typing import Annotated
from langgraph.graph.message import add_messages

class State(TypedDict):
    messages : Annotated[list,add_messages]

In [6]:
# Here what is Annotated
# in python type.Annotated is a special type qualifier that allows you to attach metadata to a datatype.
# basic syntax
from typing import Annotated

age:Annotated[int,"must be greater than 18"] = 21
print(age)

21


In [ ]:
# type.Annotated means Extra Information Attached to Type.

# normally
messages = ""
messages.list

# now
messages:Annotated[
    list,
    add_messages
    ]

# here Messages a list and use add_message as the reducer.

##### add_messages
it is a built-in LangGraph Reducer. Instead of replacing messages, it appends them. 

Example : current: hi , new: hello result: Hi Hello : this is exaclty what chatbot needs.


#### Types of Reducers (Conceptually)
Although LangGraph provides specific implementations, it's helpful to think of reducers as different merge strategies.

| Reducer Strategy | Behavior                          | Example                         |
| ---------------- | --------------------------------- | ------------------------------- |
| Replace          | New value replaces old value      | `status`, `answer`, `category`  |
| Append           | Add new items to an existing list | `messages`, `documents`, `logs` |
| Add              | Combine numeric values            | `cost`, `token_count`           |
| Custom           | Apply your own business logic     | Domain-specific fields          |


Reducers make these advanced capabilities possible:
* parallel Excecution
* multi-agent collabration
* Tool Calling in parallel
* Chat history Accumulation
* Event Logging
* Cost tracking
* Search result aggregation

#### Summary
Reducers answer one fundamental question:

If two or more updates target the same State field, how should they be combined?

Excecution flow:

``` markdown
Node A ─────┐
            │
            ▼
        State Updates
            ▲
            │
Node B ─────┘
            │
            ▼
        Reducer
            │
            ▼
     Updated Shared State
```

#### Key Takeaways
* Reducers resolve conflicts when multiple updates affect the same field.
* Different fields can have different reducer strategies.
* messages commonly uses an append reducer (add_messages).
* Fields like answer or status usually use replacement behavior.
* Reducers are defined as part of the State Schema, not inside nodes.
* Reducers are one of the key features that enable parallel execution and multi-agent workflows in LangGraph.

``` markdown
Node Returns Updates
        ↓
LangGraph Starts State Merge
        ↓
Did multiple updates affect the same field?

       / \
     No   Yes
     |      |
Normal   Use Reducer
Merge    Strategy
     \    /
      New Shared State
```

* State Merging the general process of applying updates.
* Reducers Provide special rules when the defualt merge behaviour is not sufficient.

Project : A 3-node research pipeline:

In [9]:
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, END
import operator

# State definition
class ResearchState(TypedDict):
    query:str                               # plain field : Overwrites each time.
    messages:Annotated[list,operator.add]   # reducer - Accumulates across Nodes.
    sources:Annotated[list,operator.add]    # reducer - Accumulates across Nodes.
    current_step: str                       # plain Field : Overwrtie Each time.

# print state snapshot 
def print_state(label:str,state:dict):
    print(f"\n{'='}* 50")
    print(f"  STATE AFTER: {label}")
    print(f"\n{'='}* 50")
    print(f"  query        : {state['query']}")
    print(f"  current_step : {state['current_step']}")
    print(f"  messages     : {state['messages']}")
    print(f"  sources      : {state['sources']}")
    print(f"{'='*50}\n")

# Nodes
def planner_node(state:ResearchState):
    print("\n>>> PLANNER NODE running...")
    print(f"    Input messages so far : {state['messages']}")

    return {
        "messages": [f"[planner] Plan created for query: '{state['query']}'"],
        "current_step" : "planned"
        # source is not returned so it is unchanged.
    }

def researcher_node(state: ResearchState):
    print("\n>>> RESEARCHER NODE running...")
    print(f"    Input messages so far : {state['messages']}")

    return {
        "messages": [f"[researcher] Found 3 relevant sources."],
        "sources": ["source_A.com", "source_B.com", "source_C.com"],
        "current_step": "researched"
    }


def summarizer_node(state: ResearchState):
    print("\n>>> SUMMARIZER NODE running...")
    print(f"    Input messages so far : {state['messages']}")
    print(f"    Sources available     : {state['sources']}")

    return {
        "messages": [f"[summarizer] Summary written using {len(state['sources'])} sources."],
        "current_step": "complete"
    }

# Graph Assembly
builder = StateGraph(ResearchState)

builder.add_node("planner",    planner_node)
builder.add_node("researcher", researcher_node)
builder.add_node("summarizer", summarizer_node)

builder.set_entry_point("planner")

builder.add_edge("planner",    "researcher")
builder.add_edge("researcher", "summarizer")
builder.add_edge("summarizer", END)

graph = builder.compile()

# invocation
initial_state = {
    "query": "How does LangGraph handle state?",
    "messages": [],      # starts empty — reducer will accumulate into this
    "sources": [],       # starts empty — reducer will accumulate into this
    "current_step": "start"
}

final_state = graph.invoke(initial_state)
print_state("FINAL", final_state)





>>> PLANNER NODE running...
    Input messages so far : []

>>> RESEARCHER NODE running...
    Input messages so far : ["[planner] Plan created for query: 'How does LangGraph handle state?'"]

>>> SUMMARIZER NODE running...
    Input messages so far : ["[planner] Plan created for query: 'How does LangGraph handle state?'", '[researcher] Found 3 relevant sources.']
    Sources available     : ['source_A.com', 'source_B.com', 'source_C.com']

=* 50
  STATE AFTER: FINAL

=* 50
  query        : How does LangGraph handle state?
  current_step : complete
  messages     : ["[planner] Plan created for query: 'How does LangGraph handle state?'", '[researcher] Found 3 relevant sources.', '[summarizer] Summary written using 3 sources.']
  sources      : ['source_A.com', 'source_B.com', 'source_C.com']



What This Proves

* messages starts as []. Each node returns only its own new message as a single-item list. The reducer (operator.add) concatenates them one by one as the graph flows through nodes. By the time summarizer runs, it can see everything planner and researcher wrote — because the reducer preserved it all.

* current_step has no reducer. Every node overwrites it. The final value is just whatever the last node wrote — "complete". The intermediate values "planned" and "researched" are gone.

* sources also has a reducer. Even though only researcher wrote to it, if you later added a second researcher node writing more sources, they would accumulate rather than overwrite each other.

* query is never written by any node — it stays exactly as you set it in the initial state. Nodes that don't return a key leave it completely untouched.
